<a href="https://colab.research.google.com/github/onljunior72-dot/Proyecto/blob/main/Proyectoo3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Tap this if you play on Mobile { display-mode: "form" }
%%html
<b>Ensure this audio is playing to prevent colab from shutting down, then start ColabSteam below</b><br/>
<audio autoplay="" src="https://github.com/kmille36/Colab-Cloud-Gaming/raw/refs/heads/main/silence.m4a" loop controls>

In [ ]:
# @title **🖥️ QEMU Virtual Machine - Windows Tiny ISO**
# @markdown ---
# @markdown **🔗 Pega aquí la URL de descarga directa de la ISO:**
ISO_URL = ""  # @param {type:"string"}

# @markdown **💾 Nombre del archivo ISO (opcional, déjalo vacío para autodetectar):**
ISO_FILE = ""  # @param {type:"string"}

# @markdown **🧠 RAM asignada (MB):**
RAM_MB = 9216  # @param {type:"integer"}

# @markdown **💿 Tamaño del disco virtual (GB):**
DISK_GB = 100  # @param {type:"integer"}

# @markdown ---
# @markdown **⚠️ Presiona el botón de "Play" de la celda y luego escribe la URL.**
# @markdown **La descarga comenzará automáticamente al ejecutar la celda.**

import os, time, subprocess, multiprocessing
from IPython.display import display, Javascript

# ========== Keep-Alive ==========
display(Javascript('''
 function ClickConnect(){
   console.log("Manteniendo conexión activa...");
   document.querySelector("colab-connect-button").click()
 }
 setInterval(ClickConnect, 60000)
'''))

# ========== Validación de URL ==========
if not ISO_URL or not ISO_URL.startswith("http"):
    print("❌ Debes pegar una URL válida en el campo 'ISO_URL'.")
    print("   Vuelve a ejecutar la celda después de escribir la URL.")
else:
    # ========== Instalación silenciosa ==========
    print("🔧 Preparando sistema...")
    !sudo rm -f /etc/apt/sources.list.d/*.list 2>/dev/null
    !sudo apt-get update -qq 2>/dev/null
    !sudo apt-get install -y -qq --fix-missing qemu-system-x86 qemu-utils novnc websockify wget 2>/dev/null

    if not os.path.exists("/usr/bin/qemu-system-x86_64"):
        print("❌ QEMU no se instaló.")
        raise SystemExit(1)
    print("✅ Herramientas listas.\n")

    # ========== Nombre de archivo ISO ==========
    if not ISO_FILE:
        ISO_FILE = ISO_URL.split('/')[-1].split('?')[0]
        if not ISO_FILE.endswith('.iso'):
            ISO_FILE = "custom_windows.iso"

    DISK_FILE = ISO_FILE.replace('.iso', '_disk.qcow2')
    VNC_PORT = 7
    NOVNC_PORT = 6081

    print(f"📥 URL: {ISO_URL[:100]}...")
    print(f"💾 Archivo ISO: {ISO_FILE}")
    print(f"💿 Disco virtual: {DISK_FILE}")
    print(f"📊 RAM: {RAM_MB} MB | Disco: {DISK_GB} GB")

    # ========== Descargar ISO ==========
    if not os.path.exists(ISO_FILE):
        print(f"\n📥 Descargando {ISO_FILE}...")
        !wget -q --show-progress --user-agent="Mozilla/5.0" -O {ISO_FILE} "{ISO_URL}"
        if os.path.exists(ISO_FILE) and os.path.getsize(ISO_FILE) > 1000000:
            print("✅ Descarga completada.")
        else:
            print("❌ La descarga falló. Revisa la URL.")
            raise SystemExit(1)
    else:
        print(f"✅ ISO '{ISO_FILE}' ya existe.")

    # ========== Crear disco virtual ==========
    if not os.path.exists(DISK_FILE):
        print(f"💾 Creando disco virtual de {DISK_GB} GB...")
        !qemu-img create -f qcow2 {DISK_FILE} {DISK_GB}G 2>/dev/null
    print("✅ Disco listo.")

    # ========== Limpieza ==========
    print("🧹 Liberando puertos...")
    !pkill -9 qemu 2>/dev/null
    !pkill -9 websockify 2>/dev/null
    !pkill -9 cloudflared 2>/dev/null
    !sudo fuser -k {5900 + VNC_PORT}/tcp 2>/dev/null
    !sudo fuser -k {NOVNC_PORT}/tcp 2>/dev/null
    time.sleep(2)

    # ========== Arrancar QEMU ==========
    print("🚀 Iniciando máquina virtual...")
    cpu_cores = multiprocessing.cpu_count()
    qemu_cmd = [
        "qemu-system-x86_64",
        "-m", str(RAM_MB),
        "-smp", str(cpu_cores),
        "-cpu", "max",
        "-accel", "tcg,thread=multi",
        "-machine", "type=pc",
        "-vga", "std",
        "-display", f"vnc=:{VNC_PORT}",
        "-drive", f"file={DISK_FILE},format=qcow2,index=0,media=disk",
        "-drive", f"file={ISO_FILE},format=raw,index=1,media=cdrom",
        "-boot", "order=d",
        "-usb", "-device", "usb-tablet",
        "-rtc", "base=localtime",
        "-netdev", "user,id=net0", "-device", "e1000,netdev=net0"
    ]
    subprocess.Popen(qemu_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # ========== Esperar VNC ==========
    print("⏳ Esperando servidor VNC...")
    ready = False
    for i in range(30):
        sock_check = subprocess.run(["ss", "-tlnp"], capture_output=True, text=True)
        if f"0.0.0.0:590{VNC_PORT}" in sock_check.stdout or f"127.0.0.1:590{VNC_PORT}" in sock_check.stdout:
            ready = True
            break
        time.sleep(2)

    if not ready:
        print("❌ El puerto VNC no se abrió. QEMU falló con esta ISO.")
        raise SystemExit(1)
    print("✅ Servidor VNC activo.")

    # ========== noVNC ==========
    print("🌐 Iniciando noVNC...")
    subprocess.Popen([
        "websockify", "--web", "/usr/share/novnc", str(NOVNC_PORT),
        f"localhost:590{VNC_PORT}"
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("✅ noVNC listo.")

    # ========== Cloudflare ==========
    print("⛅ Creando túnel público...")
    if not os.path.exists("cloudflared-linux-amd64.deb"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
        !sudo dpkg -i cloudflared-linux-amd64.deb 2>/dev/null

    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{NOVNC_PORT}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )

    url = None
    for line in iter(cf_proc.stdout.readline, ''):
        if "trycloudflare.com" in line:
            for part in line.split():
                if "https://" in part and "trycloudflare.com" in part:
                    url = part.strip()
                    break
        if url:
            break

    if url:
        print("\n" + "=" * 50)
        print("🌍 URL DE ACCESO:")
        print(url)
        print("=" * 50)
    else:
        print("⚠️ No se obtuvo la URL.")

    while True:
        time.sleep(60)